# FACTS Sea-Level Dashboard Generator

This notebook generates an interactive HTML dashboard from [FACTS](https://github.com/radical-collaboration/facts) sea-level projection output files.

**No installation needed beyond running these cells.**

---

### How to use
1. Run **Cell 1** to install dependencies (takes ~1 minute, only needed once per session)
2. Run **Cell 2** to connect your Google Drive
3. Edit **Cell 3** to set the path to your FACTS data
4. Run **Cell 4** to generate the dashboard
5. Run **Cell 5** to download the HTML file to your computer

> **Tip:** Run a cell by clicking it and pressing `Shift + Enter`, or click the play button on the left.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Run this once at the start of each session. Takes about 1 minute.

print('Installing dependencies...')
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'bokeh>=3.4', 'xarray>=2023.1', 'numpy>=1.24',
                'pandas>=2.0', 'netCDF4>=1.6', '--quiet'], check=True)

import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/Ttheegela/facts.plotting.dashboard/main/facts_dashboard.py',
    'facts_dashboard.py'
)

print('Done! Ready to generate dashboards.')

In [ ]:
# ── Cell 2: Connect Google Drive ─────────────────────────────────────────────
# This lets the notebook access your FACTS data stored in Google Drive.
# Click "Connect to Google Drive" when the popup appears.

from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected. Your files are at /content/drive/MyDrive/')

In [ ]:
# ── Cell 3: Set your data path ────────────────────────────────────────────────
# Edit the path below to point to your FACTS experiment folder in Google Drive.
#
# Example: if your data is in My Drive > FACTS > exp.alt.emis, set:
#   EXP_ROOT = "/content/drive/MyDrive/FACTS/exp.alt.emis"
#
# The folder should contain subfolders like:
#   coupling.ssp126/
#   coupling.ssp245/
#   coupling.ssp585/
#   ...

EXP_ROOT = "/content/drive/MyDrive/YOUR_EXPERIMENT_FOLDER"   # <-- CHANGE THIS

TITLE    = "FACTS Sea-Level Projections"   # Optional: change the dashboard title
OUTPUT   = "/content/facts_dashboard.html"

# ── Verify the path exists before running ────────────────────────────────────
import os
if not os.path.isdir(EXP_ROOT):
    print(f'ERROR: Folder not found: {EXP_ROOT}')
    print('Please check the path and re-run this cell.')
else:
    contents = os.listdir(EXP_ROOT)
    print(f'Found folder with {len(contents)} items:')
    for item in sorted(contents)[:10]:
        print(f'  {item}')
    if len(contents) > 10:
        print(f'  ... and {len(contents)-10} more')
    print('\nPath looks good. Run Cell 4 to generate the dashboard.')

In [ ]:
# ── Cell 4: Generate the dashboard ───────────────────────────────────────────
# This may take 30-60 seconds depending on the size of your dataset.

import subprocess, sys

cmd = [sys.executable, 'facts_dashboard.py',
       '--exp-root', EXP_ROOT,
       '--output',   OUTPUT,
       '--title',    TITLE]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

if result.returncode == 0:
    size_mb = os.path.getsize(OUTPUT) / (1024 * 1024)
    print(f'Dashboard ready ({size_mb:.1f} MB). Run Cell 5 to download it.')
else:
    print('Something went wrong:')
    print(result.stderr)

In [ ]:
# ── Cell 5: Download the dashboard ───────────────────────────────────────────
# Downloads facts_dashboard.html to your computer.
# Open the file in any browser — no internet connection needed.

from google.colab import files
files.download(OUTPUT)
print('Download started. Check your browser downloads folder.')

---

## Other input modes

The cells above handle the standard case (multiple SSP scenarios under one folder).
For other data layouts, use the cells below instead of Cell 3 and 4.

---

In [ ]:
# ── Alternative: Single NetCDF file ──────────────────────────────────────────
# Use this if you have one .nc file (e.g. a total sea-level file).

SINGLE_FILE = "/content/drive/MyDrive/YOUR_FILE.nc"   # <-- CHANGE THIS
OUTPUT      = "/content/facts_dashboard.html"

import subprocess, sys, os
result = subprocess.run(
    [sys.executable, 'facts_dashboard.py', '--single-nc-file', SINGLE_FILE, '--output', OUTPUT],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

In [ ]:
# ── Alternative: AR6-style confidence files ───────────────────────────────────
# Use this if you have a folder with medium_confidence/ and low_confidence/ subfolders.

CONF_ROOT   = "/content/drive/MyDrive/YOUR_CONFIDENCE_FOLDER"   # <-- CHANGE THIS
CONF_LEVEL  = "both"    # Options: "both", "medium_confidence", "low_confidence"
OUTPUT      = "/content/facts_dashboard_confidence.html"

import subprocess, sys, os
result = subprocess.run(
    [sys.executable, 'facts_dashboard.py',
     '--confidence-root',  CONF_ROOT,
     '--confidence-level', CONF_LEVEL,
     '--output',           OUTPUT],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)